In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    for item in result.get('data', []):
        if 'summary' in item:
            normalized_data.append(item)

observed_src = pd.json_normalize(normalized_data)
observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
observed_src.drop_duplicates(subset='indicator', inplace=True)
observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]


Querying owner: HTOC Org


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250618.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250617.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250616.csv']
Loaded data from 3 files.


In [28]:
import warnings

# Extract API-tagged indicators
all_filtered = []
cutoff = pd.Timestamp.utcnow()

for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if isinstance(tags_data, list):
        tags_df = pd.json_normalize(tags_data)
        api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)]
        if not api_tags.empty:
            all_tags_list = tags_df['name'].astype(str).tolist()
            api_tags['name'] = api_tags['name'].astype(str)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink', 'id']:
                api_tags[col] = row.get(col)
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            all_filtered.append(api_tags)
            warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
filtered_tags = pd.concat(all_filtered, ignore_index=True)
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Prepare for matching
filtered_tags['OpDiv'] = filtered_tags['name'].str.replace(' Splunk API', '', regex=False)
obs_subset = observed_data_df[['indicator', 'OpDiv', 'obs_date']].drop_duplicates()
recent_tags = pd.merge(filtered_tags, obs_subset, left_on=['summary', 'OpDiv'], right_on=['indicator', 'OpDiv'], how='inner')

# Filter to recent
cutoff_naive = cutoff.tz_convert(None)
recent_tags = recent_tags[
    (recent_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (pd.to_datetime(recent_tags['obs_date'], errors='coerce') >= cutoff_naive - timedelta(days=1))
].copy()

# Partner processing
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
recent_tags.drop_duplicates(subset='summary', inplace=True)


In [ ]:
# Enrich only final filtered indicators
indicator_ids = recent_tags['id'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_ids)} indicators with VirusTotalV3...")

for indicator_id in indicator_ids:
    try:
        enrich_url = f'/v3/indicators/{indicator_id}/enrich?type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)
        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['indicator_id'] = indicator_id
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched.rename(columns={'indicator_id': 'id'}, inplace=True)
    recent_tags = recent_tags.merge(df_enriched, on='id', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]
    
    recent_tags.drop(columns=['id', 'name', 'lastUsed', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'all_tags', 'techniqueId',
                              'tactics.data', 'tactics.count', 'platforms.data', 'platforms.count', 'summary', 'partner_count', 'status', 'data.id', 
                              'data.dateAdded', 'data.ownerId', 'data.ownerName', 'data.webLink', 'data.type', 'data.lastModified', 'data.rating', 'data.confidence',
                              'data.description', 'data.summary', 'data.privateFlag', 'data.active', 'data.activeLocked', 'data.ip', 'data.legacyLink', 'data.enrichment.data',
                              'data.hostName', 'data.dnsActive', 'data.whoisActive', 'data.source', 'partners', 'opDiv'], inplace=True)
    
    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 413 indicators with VirusTotalV3...


Status Code: 400
Failed API Response: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL chaturbate.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


Successfully enriched and merged 411 indicators.


In [30]:
recent_tags

,lastObserved,webLink,OpDiv,indicator,obs_date,partner,partners,vtMaliciousCount
3,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,CMS,109.70.100.2,2025-06-18,CMS,CMS,11.0
10,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,OS,78.153.140.177,2025-06-18,OS,"FDA, IHS, OS",12.0
14,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,DHA,45.95.146.59,2025-06-18,DHA,"CMS, DHA, FDA, HRSA, IHS",11.0
15,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,DHA,167.99.13.19,2025-06-18,DHA,"DHA, FDA, HRSA, IHS, OS",12.0
21,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,DHA,178.128.84.112,2025-06-18,DHA,"DHA, FDA, HRSA, OS",11.0
...,...,...,...,...,...,...,...,...
363,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,DHA,104.167.221.114,2025-06-18,DHA,"CMS, DHA, HRSA, IHS, NIH, OS",19.0
365,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,OS,118.193.72.187,2025-06-18,OS,"CMS, HHS, OS",11.0
369,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,FDA,183.224.237.233,2025-06-18,FDA,"CMS, FDA, HRSA",11.0
374,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,CMS,185.220.101.182,2025-06-18,CMS,CMS,12.0


In [21]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: CMS (39 records)


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source,vtMaliciousCount
3,6755399459033760,CMS Splunk API,2025-06-18T13:37:53Z,109.70.100.2,58,RITM0589984,Address,2025-06-16 17:42:43+00:00,2025-06-18T12:50:29Z,2025-06-18 00:00:00+00:00,...,True,False,109.70.100.2,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
14,5629499554313267,DHA Splunk API,2025-06-18T13:37:53Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,45.95.146.59,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
27,6755399447111155,DHA Splunk API,2025-06-18T13:37:53Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,91.191.209.198,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN,13.0
31,4554012,CMS Splunk API,2025-06-18T13:37:53Z,64.227.146.243,4111601,CMS Anomali ThreatStream Indicators from 01.14...,Address,2024-03-29 14:24:45+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,64.227.146.243,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
65,4036372,CMS Splunk API,2025-06-18T13:37:53Z,185.220.100.240,1280292,Our partner Health-ISAC is sharing IOCs for Bl...,Address,2021-12-15 13:22:24+00:00,2025-06-18T11:22:45Z,2025-06-18 00:00:00+00:00,...,True,False,185.220.100.240,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
67,6755399458556975,DHA Splunk API,2025-06-18T13:37:53Z,142.93.115.5,25491,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,...,True,False,142.93.115.5,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
89,5629499539158718,OS Splunk API,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,...,True,False,80.67.167.81,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
91,5629499547737365,DHA Splunk API,2025-06-18T13:37:53Z,185.169.4.150,2379467,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,...,True,False,185.169.4.150,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 20}]",NaN,NaN,NaN,NaN,20.0
100,4755432,DHA Splunk API,2025-06-18T13:37:53Z,152.32.128.214,3674721,NaN,Address,2024-07-08 15:36:44+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,...,True,False,152.32.128.214,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
102,234725,CMS Splunk API,2025-06-18T13:37:53Z,185.129.62.62,176065,TLP:WHITE\r\n\r\nJoint Analysis Report from NC...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,...,True,False,185.129.62.62,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,Department of Homeland Security \r\nNCCIC-- US...,11.0


Partner: DHA (17 records)


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source,vtMaliciousCount
14,5629499554313267,DHA Splunk API,2025-06-18T13:37:53Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,45.95.146.59,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
15,5629499554313259,DHA Splunk API,2025-06-18T13:37:53Z,167.99.13.19,76525,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,167.99.13.19,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
21,6755399458556982,DHA Splunk API,2025-06-18T13:37:53Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,178.128.84.112,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
27,6755399447111155,DHA Splunk API,2025-06-18T13:37:53Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,91.191.209.198,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN,13.0
67,6755399458556975,DHA Splunk API,2025-06-18T13:37:53Z,142.93.115.5,25491,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,...,True,False,142.93.115.5,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
91,5629499547737365,DHA Splunk API,2025-06-18T13:37:53Z,185.169.4.150,2379467,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,...,True,False,185.169.4.150,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 20}]",NaN,NaN,NaN,NaN,20.0
100,4755432,DHA Splunk API,2025-06-18T13:37:53Z,152.32.128.214,3674721,NaN,Address,2024-07-08 15:36:44+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,...,True,False,152.32.128.214,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
109,5629499542036631,DHA Splunk API,2025-06-18T13:37:53Z,193.163.125.127,302594,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:47+00:00,2025-06-18T11:22:40Z,2025-06-18 00:00:00+00:00,...,True,False,193.163.125.127,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
189,6755399459033713,DHA Splunk API,2025-06-18T13:37:53Z,71.6.232.25,272201,RITM0589984,Address,2025-06-16 17:35:09+00:00,2025-06-17T22:03:22Z,2025-06-18 00:00:00+00:00,...,True,False,71.6.232.25,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
199,5629499555060723,DHA Splunk API,2025-06-18T13:37:53Z,170.64.134.89,5029,RITM0589984,Address,2025-06-16 17:42:50+00:00,2025-06-17T15:40:26Z,2025-06-18 00:00:00+00:00,...,True,False,170.64.134.89,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0


Partner: HRSA (36 records)


...

,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source,vtMaliciousCount
339,5629499546480609,OS Splunk API,2025-06-18T12:50:29Z,195.3.221.137,324553,TASK0882701 / RITM0585661,Address,2025-06-02 18:33:29+00:00,2025-06-16T11:25:17Z,2025-06-18 00:00:00+00:00,...,True,False,195.3.221.137,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0


Partner: IHS (17 records)


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source,vtMaliciousCount
10,6755399458556984,OS Splunk API,2025-06-18T12:50:29Z,78.153.140.177,3018,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:32+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,78.153.140.177,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
14,5629499554313267,DHA Splunk API,2025-06-18T13:37:53Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,45.95.146.59,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
15,5629499554313259,DHA Splunk API,2025-06-18T13:37:53Z,167.99.13.19,76525,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,167.99.13.19,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
27,6755399447111155,DHA Splunk API,2025-06-18T13:37:53Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,91.191.209.198,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN,13.0
53,6755399448005392,OS Splunk API,2025-06-18T12:50:29Z,78.153.140.179,16448,RITM0581780/TASK0877793,Address,2025-05-19 11:39:42+00:00,2025-06-18T11:22:46Z,2025-06-18 00:00:00+00:00,...,True,False,78.153.140.179,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
67,6755399458556975,DHA Splunk API,2025-06-18T13:37:53Z,142.93.115.5,25491,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,...,True,False,142.93.115.5,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
89,5629499539158718,OS Splunk API,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,...,True,False,80.67.167.81,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
91,5629499547737365,DHA Splunk API,2025-06-18T13:37:53Z,185.169.4.150,2379467,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,...,True,False,185.169.4.150,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 20}]",NaN,NaN,NaN,NaN,20.0
100,4755432,DHA Splunk API,2025-06-18T13:37:53Z,152.32.128.214,3674721,NaN,Address,2024-07-08 15:36:44+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,...,True,False,152.32.128.214,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
189,6755399459033713,DHA Splunk API,2025-06-18T13:37:53Z,71.6.232.25,272201,RITM0589984,Address,2025-06-16 17:35:09+00:00,2025-06-17T22:03:22Z,2025-06-18 00:00:00+00:00,...,True,False,71.6.232.25,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0


Partner: OS (40 records)


,id,name,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,...,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source,vtMaliciousCount
10,6755399458556984,OS Splunk API,2025-06-18T12:50:29Z,78.153.140.177,3018,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:32+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,78.153.140.177,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
15,5629499554313259,DHA Splunk API,2025-06-18T13:37:53Z,167.99.13.19,76525,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,...,True,False,167.99.13.19,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 12}]",NaN,NaN,NaN,NaN,12.0
21,6755399458556982,DHA Splunk API,2025-06-18T13:37:53Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,178.128.84.112,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
27,6755399447111155,DHA Splunk API,2025-06-18T13:37:53Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,...,True,False,91.191.209.198,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN,13.0
35,5629499541090450,OS Splunk API,2025-06-18T12:50:29Z,104.152.52.148,613,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:48+00:00,2025-06-18T11:22:49Z,2025-06-18 00:00:00+00:00,...,True,False,104.152.52.148,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN,11.0
36,234659,OS Splunk API,2025-06-18T12:50:29Z,171.25.193.20,22240,TLP:WHITE\r\n\r\nJoint Analysis Report from NC...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:49Z,2025-06-18 00:00:00+00:00,...,True,False,171.25.193.20,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 16}]",NaN,NaN,NaN,Department of Homeland Security \r\nNCCIC-- US...,16.0
53,6755399448005392,OS Splunk API,2025-06-18T12:50:29Z,78.153.140.179,16448,RITM0581780/TASK0877793,Address,2025-05-19 11:39:42+00:00,2025-06-18T11:22:46Z,2025-06-18 00:00:00+00:00,...,True,False,78.153.140.179,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
82,4720930,OS Splunk API,2025-06-18T12:50:29Z,185.241.208.204,37628,**(U//FOUO) Additional Indicators of Compromis...,Address,2024-06-17 18:10:46+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,...,True,False,185.241.208.204,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN,13.0
89,5629499539158718,OS Splunk API,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,...,True,False,80.67.167.81,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 14}]",NaN,NaN,NaN,NaN,14.0
91,5629499547737365,DHA Splunk API,2025-06-18T13:37:53Z,185.169.4.150,2379467,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,...,True,False,185.169.4.150,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 20}]",NaN,NaN,NaN,NaN,20.0
